# HazeCast SG — Pipeline Walkthrough

Runs the same 5 steps as `README.md`'s "Running it" section, but interactively,
one stage at a time, so you can inspect intermediate output (row counts, date
ranges, gap warnings, evaluation tables) as you go.

**Before running:** place the raw source files under `data/raw/{pm25,firms,weather}/`
— see `data/README.md` for exact filenames and where to get them. This notebook
doesn't re-explain that step; it assumes the files are already there.

If you're running this via `docker compose up jupyter` (see `docker-compose.yml`),
the repo root is mounted at `/app` and `hazecast` is already importable. If you're
running JupyterLab locally instead (`pip install -r requirements.txt && jupyter lab`),
run the cell below first — it adds `src/` to the path so the same imports work
either way.

In [1]:
import sys, os, logging
from pathlib import Path

# Makes `import hazecast...` work whether this notebook is run from inside
# the Docker container (PYTHONPATH already set, see docker/pipeline.Dockerfile)
# or from a local `jupyter lab` with no PYTHONPATH set at all.
repo_root = Path.cwd()
if not (repo_root / "src" / "hazecast").exists():
    # notebook launched from notebooks/ instead of the repo root
    repo_root = repo_root.parent
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Jupyter sets the kernel's cwd to wherever the .ipynb file lives (this
# notebook's own folder, when opened via JupyterLab's file browser) --
# NOT the repo root. That silently breaks anything that writes to a
# relative path, most importantly MLflow's default local artifact store
# (`./mlruns/`), which would otherwise land inside notebooks/ instead of
# next to data/ and mlflow.db. Pin cwd to the repo root explicitly so
# every relative path in every stage below resolves the same way it does
# when running `python -m hazecast....` from the repo root.
os.chdir(repo_root)

logging.basicConfig(level=logging.INFO, force=True)

import hazecast
print("hazecast package OK, repo root:", repo_root)

hazecast package OK, repo root: /tmp/claude-0/-home-claude/b5ba3411-1186-53b7-bc68-afbadedf06d7/scratchpad/hazecast-sg


## 1. Ingest raw sources → bronze

One cell per source. Each prints the row count and date range it found — compare these against `data/README.md`'s expected ranges as a first sanity check.

In [2]:
from hazecast.ingest.pm25 import load_pm25_bronze, write_bronze as write_pm25_bronze

pm25_bronze = load_pm25_bronze()
write_pm25_bronze(pm25_bronze)
print(f"pm25: {len(pm25_bronze):,} rows, {pm25_bronze['ts'].min()} to {pm25_bronze['ts'].max()}")

pm25: 108,117 rows, 2014-04-01 01:00:00 to 2026-08-01 00:00:00


In [3]:
from hazecast.ingest.firms import load_firms_bronze, write_bronze as write_firms_bronze

firms_bronze = load_firms_bronze()
write_firms_bronze(firms_bronze)
print(f"firms: {len(firms_bronze):,} detections, {firms_bronze['date'].min()} to {firms_bronze['date'].max()}")
print(firms_bronze['region'].value_counts())

firms: 2,181,280 detections, 2014-01-01 to 2026-09-21
region
kalimantan    1255527
sumatra        925753
Name: count, dtype: int64


In [4]:
from hazecast.ingest.weather import load_weather_bronze, write_bronze as write_weather_bronze

weather_bronze = load_weather_bronze()
write_weather_bronze(weather_bronze)
for point, g in weather_bronze.groupby("point"):
    print(f"{point}: {len(g):,} rows, {g['ts'].min()} to {g['ts'].max()}")

palangka: 108,120 rows, 2014-04-01 00:00:00 to 2026-07-31 23:00:00
palembang: 108,120 rows, 2014-04-01 00:00:00 to 2026-07-31 23:00:00
pontianak: 108,120 rows, 2014-04-01 00:00:00 to 2026-07-31 23:00:00
riau: 108,120 rows, 2014-04-01 00:00:00 to 2026-07-31 23:00:00
singapore: 108,120 rows, 2014-04-01 00:00:00 to 2026-07-31 23:00:00


## 2. Bronze → Silver

Aligns each source to a full hourly (or daily, for fire counts) grid and flags gaps — this is where the pipeline surfaces data-quality issues rather than hiding them.

In [5]:
from hazecast.bronze_to_silver import pm25_silver, weather_silver, firms_daily_silver, write_silver

pm25_s = pm25_silver()
write_silver(pm25_s, "pm25")
print(f"silver/pm25: {len(pm25_s):,} rows")

weather_s = weather_silver()
write_silver(weather_s, "weather")
print(f"silver/weather: {len(weather_s):,} rows")

firms_daily_s = firms_daily_silver()
write_silver(firms_daily_s, "firms_daily")
print(f"silver/firms_daily: {len(firms_daily_s):,} days, {int(firms_daily_s['data_gap'].sum())} flagged as likely data gaps")

silver/pm25: 108,120 rows


silver/weather: 540,600 rows


silver/firms_daily: 4,647 days, 73 flagged as likely data gaps


## 3. Silver → Gold

Builds the one-row-per-day feature + label table. See `silver_to_gold.py`'s docstring and `tests/test_pipeline_smoke.py` for how leak-freedom is enforced and tested.

In [6]:
from hazecast.silver_to_gold import build_gold, write_gold

gold = build_gold(pm25_s, weather_s, firms_daily_s)
write_gold(gold)
print(f"gold: {gold.shape[0]:,} rows x {gold.shape[1]} columns, {gold['date'].min()} to {gold['date'].max()}")
gold.tail()

gold: 4,506 rows x 61 columns, 2014-04-01 00:00:00 to 2026-08-01 00:00:00


,date,fire_sumatra_lag0,fire_sumatra_lag1,fire_sumatra_lag2,fire_sumatra_lag3,fire_sumatra_roll3,fire_sumatra_roll7,fire_sumatra_log_lag0,fire_kalimantan_lag0,fire_kalimantan_lag1,...,today_west,today_central,today_national,label_north,label_south,label_east,label_west,label_central,label_national,label_low_coverage
4501,2026-07-28,169,98.0,93.0,64.0,360.0,667.0,5.135798,268,114.0,...,17.583333,22.166667,16.658333,13.666667,12.750000,17.000000,16.083333,19.833333,15.866667,False
4502,2026-07-29,112,169.0,98.0,93.0,379.0,720.0,4.727388,189,268.0,...,16.083333,19.833333,15.866667,13.541667,11.250000,14.125000,15.416667,18.291667,14.525000,False
4503,2026-07-30,69,112.0,169.0,98.0,350.0,686.0,4.248495,344,189.0,...,15.416667,18.291667,14.525000,14.208333,13.958333,14.333333,15.500000,18.958333,15.391667,False
4504,2026-07-31,104,69.0,112.0,169.0,285.0,709.0,4.653960,491,344.0,...,15.500000,18.958333,15.391667,19.000000,17.000000,21.000000,10.000000,22.000000,17.800000,True
4505,2026-08-01,138,104.0,69.0,112.0,311.0,783.0,4.934474,699,491.0,...,10.000000,22.000000,17.800000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Train — leave-one-season-out evaluation

Compares 4 candidates per region, holding out one full haze season at a time (see `hazecast.config.HAZE_SEASONS`): `persistence`, `linreg`, `xgboost` (plain MAE-minimizing regression), and `xgboost_q85` (quantile regression targeting the 85th percentile — built specifically to fix `xgboost`'s tendency to under-predict rare elevated days). Read `README.md`'s **Results** section for the full writeup of why `xgboost_q85` exists and how `XGB_QUANTILE_ALPHA` was chosen — in short, MAE alone hides how each model does on elevated days specifically, which is why `recall_elevated` / `precision_elevated` are broken out below too.

In [7]:
from hazecast.train import evaluate_all, train_final_models
from hazecast.config import MODELS_DIR

eval_df = evaluate_all(gold)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
eval_df.to_csv(MODELS_DIR / "eval_summary.csv", index=False)

pivot = eval_df.pivot_table(index=["target", "season"], columns="model", values="mae")
pivot.round(2)

2026/10/14 18:09:41 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /usr/local/lib/python3.11/dist-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


INFO:hazecast.train:Evaluating leave-one-season-out on: ['2015', '2019']


model            linreg  persistence  xgboost  xgboost_q85
target   season                                           
central  2015     13.68        15.76    15.50        15.67
         2019      5.41         6.04     4.73         6.53
east     2015     16.06        18.79    18.31        17.80
         2019      6.41         6.89     5.53         7.65
national 2015     15.74        17.92    18.58        17.35
         2019      5.84         6.44     5.05         7.01
north    2015     15.70        17.50    18.94        17.30
         2019      5.73         6.31     5.02         6.85
south    2015     17.66        20.09    21.08        19.32
         2019      6.52         7.36     5.74         7.53
west     2015     16.93        18.67    21.03        19.44
         2019      6.13         6.68     5.55         8.03

In [8]:
# Same table, but for the metric that actually matters for this use case:
# recall on elevated (>=56 ug/m3) days. A model can have a great average
# MAE while still missing most of the days you'd actually want a warning for.
recall_pivot = eval_df.pivot_table(index=["target", "season"], columns="model", values="recall_elevated")
recall_pivot.round(2)

model            linreg  persistence  xgboost  xgboost_q85
target   season                                           
central  2015      0.56         0.59     0.07         0.15
         2019      0.50         0.00     0.00         1.00
east     2015      0.57         0.61     0.21         0.39
         2019      0.25         0.25     0.50         0.50
national 2015      0.61         0.68     0.13         0.45
         2019      0.75         0.25     0.50         1.00
north    2015      0.53         0.66     0.06         0.16
         2019      0.50         0.00     0.50         1.00
south    2015      0.72         0.69     0.25         0.72
         2019      0.71         0.43     0.86         0.86
west     2015      0.63         0.69     0.17         0.37
         2019      0.75         0.25     1.00         0.75

In [9]:
# Fits the final XGBoost model per region on ALL available labelled rows
# (not just the LOSO folds) and saves it to data/models/ -- this is what
# score_latest() below actually loads.
model_paths = train_final_models(gold)
model_paths

{'north': PosixPath('/tmp/claude-0/-home-claude/b5ba3411-1186-53b7-bc68-afbadedf06d7/scratchpad/hazecast-sg/data/models/xgb_north.json'),
 'south': PosixPath('/tmp/claude-0/-home-claude/b5ba3411-1186-53b7-bc68-afbadedf06d7/scratchpad/hazecast-sg/data/models/xgb_south.json'),
 'east': PosixPath('/tmp/claude-0/-home-claude/b5ba3411-1186-53b7-bc68-afbadedf06d7/scratchpad/hazecast-sg/data/models/xgb_east.json'),
 'west': PosixPath('/tmp/claude-0/-home-claude/b5ba3411-1186-53b7-bc68-afbadedf06d7/scratchpad/hazecast-sg/data/models/xgb_west.json'),
 'central': PosixPath('/tmp/claude-0/-home-claude/b5ba3411-1186-53b7-bc68-afbadedf06d7/scratchpad/hazecast-sg/data/models/xgb_central.json'),
 'national': PosixPath('/tmp/claude-0/-home-claude/b5ba3411-1186-53b7-bc68-afbadedf06d7/scratchpad/hazecast-sg/data/models/xgb_national.json')}

## 5. Score the latest available day

In [10]:
from hazecast.score import score_latest, write_forecast

forecast = score_latest()
write_forecast(forecast)
forecast

,scored_at,as_of_date,forecast_date,region,forecast_pm25,elevated_flag
0,2026-10-14T10:10:19.554122+00:00,2026-07-31,2026-08-01,north,17.4,False
1,2026-10-14T10:10:19.575822+00:00,2026-07-31,2026-08-01,south,15.8,False
2,2026-10-14T10:10:19.597475+00:00,2026-07-31,2026-08-01,east,19.9,False
3,2026-10-14T10:10:19.618164+00:00,2026-07-31,2026-08-01,west,18.0,False
4,2026-10-14T10:10:19.643412+00:00,2026-07-31,2026-08-01,central,21.6,False
5,2026-10-14T10:10:19.671508+00:00,2026-07-31,2026-08-01,national,17.8,False


## Smoke tests (optional)

Fast tests on synthetic fixtures — no real data needed. Good to re-run after any code change to `bronze_to_silver.py` or `silver_to_gold.py` in particular, since these guard the leak-free label construction.

In [11]:
import subprocess, os
env = {**os.environ, "PYTHONPATH": src_path}
result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-v"],
    cwd=repo_root, capture_output=True, text=True, env=env,
)
print(result.stdout)
print(result.stderr)

============================= test session starts ==============================
platform linux -- Python 3.11.15, pytest-9.1.1, pluggy-1.6.0 -- /usr/bin/python
cachedir: .pytest_cache
rootdir: /tmp/claude-0/-home-claude/b5ba3411-1186-53b7-bc68-afbadedf06d7/scratchpad/hazecast-sg
plugins: anyio-4.13.0
collecting ... collected 3 items

tests/test_pipeline_smoke.py::test_pm25_silver_fills_full_hourly_grid PASSED [ 33%]
tests/test_pipeline_smoke.py::test_firms_daily_silver_flags_long_zero_runs PASSED [ 66%]
tests/test_pipeline_smoke.py::test_gold_labels_are_leak_free_and_shifted_by_one_day PASSED [100%]

============================== 3 passed in 0.42s ===============================


